# Model Serving metrics collector

This notebook collects Prometheus/OpenMetrics exports from Model Serving endpoints, persists raw and normalized history in Delta, and maintains dashboard-ready rate and percentile views. It is normally executed by the recurring job created by the setup notebook.

In [ ]:
import json
import logging
import math
import re
import time
import uuid
from datetime import datetime, timezone
from importlib.metadata import version
from typing import Any
from urllib.parse import quote

from databricks.sdk import WorkspaceClient
from prometheus_client.parser import text_string_to_metric_families
from pyspark.sql import types as T

"""Collect Model Serving Prometheus/OpenMetrics exports into Delta.

Configuration is kept in this first cell. Normally this notebook stays alive in
the continuous job created by model_serving_metrics_setup.ipynb.
INCLUDE_ENDPOINT_NAMES optionally restricts collection. An empty list discovers
all non-foundation endpoints in the workspace. Job parameters override these defaults.
"""

INCLUDE_ENDPOINT_NAMES: list[str] = []
STORAGE_CATALOG = spark.sql("SELECT current_catalog()").first()[0]
STORAGE_SCHEMA = "model_serving_observability"
POLL_INTERVAL_SECONDS = 60
ENDPOINT_REFRESH_CYCLES = 60

LOG = logging.getLogger("serving_metrics_collector")
workspace = WorkspaceClient()
WORKSPACE_HOST = workspace.config.host.rstrip("/")

## Resolve collection parameters

Apply job parameter overrides, discover all endpoints when none are supplied, validate identifiers, and derive table names.

In [ ]:
def _parse_endpoint_names(value: str | list[str] | None) -> list[str]:
    """Parse one endpoint name, comma-separated names, or a JSON string array."""
    if isinstance(value, list):
        return [name.strip() for name in value if name.strip()]
    text = (value or "").strip()
    if text.startswith("["):
        parsed = json.loads(text)
        if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):
            raise ValueError("JSON endpoint_names must be an array of strings")
        return [item.strip() for item in parsed if item.strip()]
    return [item.strip() for item in text.split(",") if item.strip()]


def _is_foundation_endpoint(endpoint: Any) -> bool:
    """Return whether an endpoint routes to a system.ai foundation model."""
    endpoint_dict = endpoint.as_dict()
    config = endpoint_dict.get("config") or {}
    entities = config.get("served_entities") or config.get("served_models") or []
    return any(
        str(entity.get("entity_name") or entity.get("model_name") or "").startswith("system.ai.")
        for entity in entities
    )


def _discover_endpoint_names(include_names: list[str]) -> list[str]:
    """Discover non-foundation endpoints, optionally restricted by an include list."""
    endpoints = [
        endpoint
        for endpoint in workspace.serving_endpoints.list()
        if endpoint.name and not _is_foundation_endpoint(endpoint)
    ]
    if include_names:
        included = set(include_names)
        endpoints = [endpoint for endpoint in endpoints if endpoint.name in included]
    return sorted(endpoint.name for endpoint in endpoints if endpoint.name)


def _sql_string_list(names: list[str]) -> str:
    """Render endpoint names as a SQL IN-list."""
    return ", ".join("'" + name.replace("'", "''") + "'" for name in names)


widget_defaults = {
    "include_endpoint_names": json.dumps(INCLUDE_ENDPOINT_NAMES),
    "storage_catalog": STORAGE_CATALOG,
    "storage_schema": STORAGE_SCHEMA,
    "poll_interval_seconds": str(POLL_INTERVAL_SECONDS),
}
for widget_name, default_value in widget_defaults.items():
    try:
        dbutils.widgets.get(widget_name)
    except Exception:
        dbutils.widgets.text(widget_name, default_value)

STORAGE_CATALOG = dbutils.widgets.get("storage_catalog").strip()
STORAGE_SCHEMA = dbutils.widgets.get("storage_schema").strip()
POLL_INTERVAL_SECONDS = int(dbutils.widgets.get("poll_interval_seconds"))
INCLUDE_ENDPOINT_NAMES = _parse_endpoint_names(
    dbutils.widgets.get("include_endpoint_names")
)
ENDPOINT_NAMES = _discover_endpoint_names(INCLUDE_ENDPOINT_NAMES)

identifier_pattern = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
for identifier_name, identifier_value in {
    "STORAGE_CATALOG": STORAGE_CATALOG,
    "STORAGE_SCHEMA": STORAGE_SCHEMA,
}.items():
    if not identifier_pattern.fullmatch(identifier_value):
        raise ValueError(
            f"{identifier_name} must contain only letters, numbers, and underscores: "
            f"{identifier_value!r}"
        )

if not ENDPOINT_NAMES:
    raise RuntimeError("The workspace contains no Model Serving endpoints")
if POLL_INTERVAL_SECONDS < 10:
    raise ValueError("POLL_INTERVAL_SECONDS must be at least 10")
if ENDPOINT_REFRESH_CYCLES < 1:
    raise ValueError("ENDPOINT_REFRESH_CYCLES must be at least 1")

ENDPOINT_NAMES_SQL = _sql_string_list(ENDPOINT_NAMES)
TABLE_PREFIX = f"{STORAGE_CATALOG}.{STORAGE_SCHEMA}"
ENDPOINT_SNAPSHOTS_TABLE = f"{TABLE_PREFIX}.model_serving_endpoint_snapshots"
RAW_METRICS_TABLE = f"{TABLE_PREFIX}.model_serving_metrics_raw"
METRIC_SAMPLES_TABLE = f"{TABLE_PREFIX}.model_serving_metric_samples"
COUNTER_RATES_VIEW = f"{TABLE_PREFIX}.model_serving_counter_rates"
HISTOGRAM_PERCENTILES_VIEW = f"{TABLE_PREFIX}.model_serving_histogram_percentiles"
DASHBOARD_SERIES_VIEW = f"{TABLE_PREFIX}.model_serving_dashboard_series"

LOG.info(
    "Continuously collecting %d endpoint candidate(s) into %s every %d second(s)",
    len(ENDPOINT_NAMES),
    TABLE_PREFIX,
    POLL_INTERVAL_SECONDS,
)

## Create Delta storage

Create the target schema and Delta tables used by idempotent source-key MERGEs.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TABLE_PREFIX}")

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {ENDPOINT_SNAPSHOTS_TABLE} (
      scrape_id STRING NOT NULL,
      scraped_at TIMESTAMP NOT NULL,
      workspace_host STRING NOT NULL,
      endpoint_name STRING NOT NULL,
      state_ready STRING,
      state_config_update STRING,
      endpoint_json STRING NOT NULL
    ) USING DELTA
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {RAW_METRICS_TABLE} (
      scrape_id STRING NOT NULL,
      scraped_at TIMESTAMP NOT NULL,
      workspace_host STRING NOT NULL,
      endpoint_name STRING NOT NULL,
      exposition_text STRING NOT NULL,
      metric_family_count BIGINT NOT NULL,
      sample_count BIGINT NOT NULL,
      parser_library STRING NOT NULL,
      parser_version STRING NOT NULL
    ) USING DELTA
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {METRIC_SAMPLES_TABLE} (
      scrape_id STRING NOT NULL,
      scraped_at TIMESTAMP NOT NULL,
      workspace_host STRING NOT NULL,
      endpoint_name STRING NOT NULL,
      metric_family STRING NOT NULL,
      metric_type STRING NOT NULL,
      metric_help STRING,
      sample_name STRING NOT NULL,
      value DOUBLE NOT NULL,
      sample_timestamp TIMESTAMP,
      labels MAP<STRING, STRING> NOT NULL,
      labels_json STRING NOT NULL,
      served_entity_name STRING,
      replica_name STRING,
      bucket_upper_bound STRING,
      bucket_upper_bound_value DOUBLE
    ) USING DELTA
    """
)

## Collect endpoint metrics

Fetch each endpoint's Prometheus exposition, normalize finite samples, and append endpoint, raw, and sample records to Delta.

In [ ]:
endpoint_snapshot_schema = T.StructType(
    [
        T.StructField("scrape_id", T.StringType(), False),
        T.StructField("scraped_at", T.TimestampType(), False),
        T.StructField("workspace_host", T.StringType(), False),
        T.StructField("endpoint_name", T.StringType(), False),
        T.StructField("state_ready", T.StringType(), True),
        T.StructField("state_config_update", T.StringType(), True),
        T.StructField("endpoint_json", T.StringType(), False),
    ]
)

raw_metrics_schema = T.StructType(
    [
        T.StructField("scrape_id", T.StringType(), False),
        T.StructField("scraped_at", T.TimestampType(), False),
        T.StructField("workspace_host", T.StringType(), False),
        T.StructField("endpoint_name", T.StringType(), False),
        T.StructField("exposition_text", T.StringType(), False),
        T.StructField("metric_family_count", T.LongType(), False),
        T.StructField("sample_count", T.LongType(), False),
        T.StructField("parser_library", T.StringType(), False),
        T.StructField("parser_version", T.StringType(), False),
    ]
)

metric_sample_schema = T.StructType(
    [
        T.StructField("scrape_id", T.StringType(), False),
        T.StructField("scraped_at", T.TimestampType(), False),
        T.StructField("workspace_host", T.StringType(), False),
        T.StructField("endpoint_name", T.StringType(), False),
        T.StructField("metric_family", T.StringType(), False),
        T.StructField("metric_type", T.StringType(), False),
        T.StructField("metric_help", T.StringType(), True),
        T.StructField("sample_name", T.StringType(), False),
        T.StructField("value", T.DoubleType(), False),
        T.StructField("sample_timestamp", T.TimestampType(), True),
        T.StructField("labels", T.MapType(T.StringType(), T.StringType()), False),
        T.StructField("labels_json", T.StringType(), False),
        T.StructField("served_entity_name", T.StringType(), True),
        T.StructField("replica_name", T.StringType(), True),
        T.StructField("bucket_upper_bound", T.StringType(), True),
        T.StructField("bucket_upper_bound_value", T.DoubleType(), True),
    ]
)


def utc_timestamp(value: Any) -> datetime | None:
    if value is None:
        return None

    if hasattr(value, "sec"):
        seconds = float(value.sec) + float(getattr(value, "nsec", 0)) / 1_000_000_000
    else:
        seconds = float(value)
        if abs(seconds) > 100_000_000_000:
            seconds /= 1_000

    return datetime.fromtimestamp(seconds, tz=timezone.utc).replace(tzinfo=None)


def served_entity_name(labels: dict[str, str]) -> str | None:
    for label_name in (
        "served_entity_name",
        "served_model_name",
        "served_model",
        "model_name",
    ):
        if labels.get(label_name):
            return labels[label_name]
    return None


def bucket_upper_bound(labels: dict[str, str]) -> tuple[str | None, float | None]:
    upper_bound = labels.get("le")
    if upper_bound is None:
        return None, None
    if upper_bound.lower() in {"+inf", "inf", "infinity", "+infinity"}:
        return upper_bound, None
    return upper_bound, float(upper_bound)


def read_metrics_text(payload: Any) -> str:
    """Decode Prometheus/OpenMetrics exposition text from a raw HTTP body."""
    if isinstance(payload, dict):
        payload = payload.get("contents")
    if payload is None:
        raise RuntimeError("The metrics API returned no contents")
    if hasattr(payload, "read"):
        payload = payload.read()
    if isinstance(payload, str):
        return payload
    return bytes(payload).decode("utf-8")


def fetch_metrics_text(workspace: WorkspaceClient, endpoint_name: str) -> str:
    """GET serving-endpoint metrics as Prometheus text, not JSON.

    ``/api/2.0/serving-endpoints/{name}/metrics`` returns ``text/plain``.
    ``WorkspaceClient.export_metrics`` on some SDK versions JSON-decodes that
    body and raises ``JSONDecodeError``. Request the raw stream and parse it
    with prometheus-client instead.
    """
    response = workspace.api_client.do(
        "GET",
        f"/api/2.0/serving-endpoints/{quote(endpoint_name, safe='')}/metrics",
        headers={"Accept": "text/plain"},
        raw=True,
    )
    return read_metrics_text(response)


def _merge_rows(
    rows: list[tuple[Any, ...]],
    schema: T.StructType,
    table_name: str,
    key_columns: tuple[str, ...],
) -> None:
    """Merge rows by stable source keys so retries do not create duplicates."""
    if not rows:
        return
    source_view = f"_metrics_merge_{uuid.uuid4().hex}"
    spark.createDataFrame(rows, schema).createOrReplaceTempView(source_view)
    predicate = " AND ".join(
        f"target.`{column}` <=> source.`{column}`" for column in key_columns
    )
    try:
        spark.sql(
            f"""
            MERGE INTO {table_name} AS target
            USING {source_view} AS source
            ON {predicate}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            """
        )
    finally:
        spark.catalog.dropTempView(source_view)


def scrape_endpoint(workspace: WorkspaceClient, endpoint_name: str) -> dict[str, int | str]:
    retrieved_at = datetime.now(timezone.utc).replace(tzinfo=None)

    endpoint = workspace.serving_endpoints.get(endpoint_name)
    endpoint_dict = endpoint.as_dict()
    endpoint_state = endpoint_dict.get("state") or {}

    exposition_text = fetch_metrics_text(workspace, endpoint_name)
    metric_families = list(text_string_to_metric_families(exposition_text))
    source_timestamps = [
        timestamp
        for family in metric_families
        for sample in family.samples
        if (timestamp := utc_timestamp(sample.timestamp)) is not None
    ]
    scraped_at = (
        max(source_timestamps)
        if source_timestamps
        else retrieved_at.replace(second=0, microsecond=0)
    )
    scrape_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"{WORKSPACE_HOST}/{endpoint_name}/{scraped_at.isoformat()}",
        )
    )

    sample_rows: list[tuple[Any, ...]] = []
    for family in metric_families:
        for sample in family.samples:
            labels = {str(key): str(value) for key, value in sample.labels.items()}
            labels_json = json.dumps(dict(sorted(labels.items())), separators=(",", ":"))
            upper_bound, upper_bound_value = bucket_upper_bound(labels)
            value = float(sample.value)
            if not math.isfinite(value):
                continue

            sample_rows.append(
                (
                    scrape_id,
                    scraped_at,
                    WORKSPACE_HOST,
                    endpoint_name,
                    family.name,
                    family.type,
                    family.documentation or None,
                    sample.name,
                    value,
                    utc_timestamp(sample.timestamp),
                    labels,
                    labels_json,
                    served_entity_name(labels),
                    labels.get("replica") or labels.get("replica_name"),
                    upper_bound,
                    upper_bound_value,
                )
            )

    if not sample_rows:
        raise RuntimeError("The metrics API response contained no finite samples")

    endpoint_row = (
        scrape_id,
        scraped_at,
        WORKSPACE_HOST,
        endpoint_name,
        endpoint_state.get("ready"),
        endpoint_state.get("config_update"),
        json.dumps(endpoint_dict, sort_keys=True, default=str),
    )
    raw_row = (
        scrape_id,
        scraped_at,
        WORKSPACE_HOST,
        endpoint_name,
        exposition_text,
        len(metric_families),
        len(sample_rows),
        "prometheus-client",
        version("prometheus-client"),
    )

    _merge_rows(
        [endpoint_row],
        endpoint_snapshot_schema,
        ENDPOINT_SNAPSHOTS_TABLE,
        ("scrape_id",),
    )
    _merge_rows(
        [raw_row],
        raw_metrics_schema,
        RAW_METRICS_TABLE,
        ("scrape_id",),
    )
    _merge_rows(
        sample_rows,
        metric_sample_schema,
        METRIC_SAMPLES_TABLE,
        ("scrape_id", "sample_name", "labels_json"),
    )

    return {
        "scrape_id": scrape_id,
        "endpoint_name": endpoint_name,
        "metric_families": len(metric_families),
        "samples": len(sample_rows),
    }


def _collect_cycle(endpoint_names: list[str]) -> tuple[list[dict[str, int | str]], dict[str, str]]:
    """Collect one snapshot from each endpoint candidate."""
    results: list[dict[str, int | str]] = []
    failures: dict[str, str] = {}
    for endpoint_name in endpoint_names:
        try:
            result = scrape_endpoint(workspace, endpoint_name)
            results.append(result)
            LOG.info("Collected metrics: %s", json.dumps(result, sort_keys=True))
        except Exception as error:
            failures[endpoint_name] = str(error)
            LOG.warning("Metrics unavailable for %s: %s", endpoint_name, error)
    return results, failures


def _sleep_until_next_boundary(interval_seconds: int) -> None:
    """Sleep until the next wall-clock interval boundary without accumulating drift."""
    now = time.time()
    next_boundary = (int(now) // interval_seconds + 1) * interval_seconds
    time.sleep(max(0.0, next_boundary - now))

## Refresh analytical views

Create the counter-rate, histogram-percentile, and dashboard-series views from the collected samples.

In [ ]:
spark.sql(
    f"""
    CREATE OR REPLACE VIEW {COUNTER_RATES_VIEW} AS
    WITH ordered AS (
      SELECT
        scraped_at,
        workspace_host,
        endpoint_name,
        served_entity_name,
        sample_name,
        labels,
        labels_json,
        value,
        LAG(value) OVER (
          PARTITION BY workspace_host, endpoint_name, served_entity_name, sample_name, labels_json
          ORDER BY scraped_at
        ) AS previous_value,
        LAG(scraped_at) OVER (
          PARTITION BY workspace_host, endpoint_name, served_entity_name, sample_name, labels_json
          ORDER BY scraped_at
        ) AS previous_scraped_at
      FROM {METRIC_SAMPLES_TABLE}
      WHERE sample_name IN (
        'request_count_total',
        'request_4xx_count_total',
        'request_5xx_count_total'
      )
    ), intervals AS (
      SELECT
        *,
        unix_timestamp(scraped_at) - unix_timestamp(previous_scraped_at) AS elapsed_seconds
      FROM ordered
    )
    SELECT
      scraped_at,
      workspace_host,
      endpoint_name,
      served_entity_name,
      sample_name,
      labels,
      CASE
        WHEN previous_value IS NULL OR elapsed_seconds <= 0 THEN NULL
        WHEN value >= previous_value THEN (value - previous_value) / elapsed_seconds
        ELSE value / elapsed_seconds
      END AS value_per_second
    FROM intervals
    """
)

spark.sql(
    f"""
    CREATE OR REPLACE VIEW {HISTOGRAM_PERCENTILES_VIEW} AS
    WITH buckets AS (
      SELECT
        scraped_at,
        workspace_host,
        endpoint_name,
        served_entity_name,
        metric_family,
        labels,
        to_json(map_filter(labels, (key, value) -> key <> 'le')) AS series_labels_json,
        bucket_upper_bound,
        bucket_upper_bound_value,
        value,
        LAG(value) OVER (
          PARTITION BY
            workspace_host,
            endpoint_name,
            served_entity_name,
            metric_family,
            to_json(map_filter(labels, (key, value) -> key <> 'le')),
            bucket_upper_bound
          ORDER BY scraped_at
        ) AS previous_value,
        LAG(scraped_at) OVER (
          PARTITION BY
            workspace_host,
            endpoint_name,
            served_entity_name,
            metric_family,
            to_json(map_filter(labels, (key, value) -> key <> 'le')),
            bucket_upper_bound
          ORDER BY scraped_at
        ) AS previous_scraped_at
      FROM {METRIC_SAMPLES_TABLE}
      WHERE endswith(sample_name, '_bucket')
    ), interval_buckets AS (
      SELECT
        *,
        CASE
          WHEN previous_value IS NULL OR previous_scraped_at IS NULL THEN NULL
          WHEN value >= previous_value THEN value - previous_value
          ELSE value
        END AS bucket_count_delta
      FROM buckets
    ), totals AS (
      SELECT
        *,
        MAX(
          CASE
            WHEN lower(bucket_upper_bound) IN ('+inf', 'inf', 'infinity', '+infinity')
            THEN bucket_count_delta
          END
        ) OVER (
          PARTITION BY
            scraped_at,
            workspace_host,
            endpoint_name,
            served_entity_name,
            metric_family,
            series_labels_json
        ) AS total_count_delta
      FROM interval_buckets
    ), percentiles AS (
      SELECT 0.50 AS quantile, 'p50' AS percentile
      UNION ALL
      SELECT 0.99 AS quantile, 'p99' AS percentile
    ), candidates AS (
      SELECT
        totals.*,
        percentiles.quantile,
        percentiles.percentile,
        ROW_NUMBER() OVER (
          PARTITION BY
            scraped_at,
            workspace_host,
            endpoint_name,
            served_entity_name,
            metric_family,
            series_labels_json,
            percentiles.percentile
          ORDER BY bucket_upper_bound_value
        ) AS candidate_rank
      FROM totals
      CROSS JOIN percentiles
      WHERE
        bucket_upper_bound_value IS NOT NULL
        AND total_count_delta > 0
        AND bucket_count_delta >= total_count_delta * percentiles.quantile
    )
    SELECT
      scraped_at,
      workspace_host,
      endpoint_name,
      served_entity_name,
      metric_family,
      percentile,
      quantile,
      bucket_upper_bound_value AS approximate_value,
      total_count_delta
    FROM candidates
    WHERE candidate_rank = 1
    """
)

spark.sql(
    f"""
    CREATE OR REPLACE VIEW {DASHBOARD_SERIES_VIEW} AS
    SELECT
      scraped_at,
      workspace_host,
      endpoint_name,
      served_entity_name,
      CASE sample_name
        WHEN 'request_count_total' THEN 'Response rate (per second)'
        WHEN 'request_4xx_count_total' THEN 'Response error rates (per second)'
        WHEN 'request_5xx_count_total' THEN 'Response error rates (per second)'
      END AS chart_name,
      CASE sample_name
        WHEN 'request_count_total' THEN 'response rate'
        WHEN 'request_4xx_count_total' THEN '4XX errors per second'
        WHEN 'request_5xx_count_total' THEN '5XX errors per second'
      END AS series_name,
      value_per_second AS value
    FROM {COUNTER_RATES_VIEW}
    WHERE value_per_second IS NOT NULL

    UNION ALL

    SELECT
      scraped_at,
      workspace_host,
      endpoint_name,
      served_entity_name,
      CASE metric_family
        WHEN 'request_latency_ms' THEN 'Latency (ms)'
        WHEN 'model_prediction_latency_ms' THEN 'Model inference latency (ms)'
        WHEN 'model_queue_time_ms' THEN 'Request queue wait time (ms)'
      END AS chart_name,
      percentile AS series_name,
      approximate_value AS value
    FROM {HISTOGRAM_PERCENTILES_VIEW}
    WHERE metric_family IN (
      'request_latency_ms',
      'model_prediction_latency_ms',
      'model_queue_time_ms'
    )

    UNION ALL

    SELECT
      scraped_at,
      workspace_host,
      endpoint_name,
      served_entity_name,
      CASE sample_name
        WHEN 'cpu_usage_percentage' THEN 'CPU Usage (%)'
        WHEN 'mem_usage_percentage' THEN 'Memory Usage (%)'
        WHEN 'provisioned_concurrent_requests_total' THEN 'Provisioned Concurrency'
      END AS chart_name,
      CASE sample_name
        WHEN 'cpu_usage_percentage' THEN 'average across replicas'
        WHEN 'mem_usage_percentage' THEN 'average across replicas'
        WHEN 'provisioned_concurrent_requests_total' THEN 'provisioned concurrency'
      END AS series_name,
      value
    FROM {METRIC_SAMPLES_TABLE}
    WHERE sample_name IN (
      'cpu_usage_percentage',
      'mem_usage_percentage',
      'provisioned_concurrent_requests_total'
    )
    """
)

## Recent normalized samples

In [ ]:
display(
    spark.sql(
        f"""
        SELECT *
        FROM {METRIC_SAMPLES_TABLE}
        WHERE endpoint_name IN ({ENDPOINT_NAMES_SQL})
        ORDER BY scraped_at DESC, metric_family, sample_name
        LIMIT 500
        """
    )
)

## Recent dashboard series

Inspect the rolling 24-hour values exposed to the AI/BI dashboard. Counter rates and histogram percentiles appear after at least two snapshots.

In [ ]:
display(
    spark.sql(
        f"""
        SELECT
          scraped_at,
          endpoint_name,
          chart_name,
          series_name,
          value
        FROM {DASHBOARD_SERIES_VIEW}
        WHERE endpoint_name IN ({ENDPOINT_NAMES_SQL})
          AND scraped_at >= timestampadd(HOUR, -24, current_timestamp())
          AND chart_name IS NOT NULL
        ORDER BY scraped_at DESC, chart_name, series_name
        LIMIT 500
        """
    )
)

## Run continuous collection

Scrape every metrics-capable endpoint on aligned one-minute boundaries. Rediscover workspace endpoints periodically without dropping previously successful endpoints after transient failures.

In [ ]:
supported_endpoint_names: set[str] = set()
cycle_index = 0

while True:
    discovery_cycle = (
        not supported_endpoint_names
        or cycle_index % ENDPOINT_REFRESH_CYCLES == 0
    )
    endpoint_candidates = (
        _discover_endpoint_names(INCLUDE_ENDPOINT_NAMES)
        if discovery_cycle
        else sorted(supported_endpoint_names)
    )
    cycle_started = time.monotonic()
    cycle_results, cycle_failures = _collect_cycle(endpoint_candidates)
    if discovery_cycle:
        supported_endpoint_names.intersection_update(endpoint_candidates)
    supported_endpoint_names.update(
        str(result["endpoint_name"]) for result in cycle_results
    )

    cycle_duration = time.monotonic() - cycle_started
    LOG.info(
        "Collection cycle %d completed in %.1f seconds: %d succeeded, %d failed",
        cycle_index,
        cycle_duration,
        len(cycle_results),
        len(cycle_failures),
    )
    if cycle_duration > POLL_INTERVAL_SECONDS:
        LOG.warning(
            "Collection cycle exceeded the %d-second interval by %.1f seconds",
            POLL_INTERVAL_SECONDS,
            cycle_duration - POLL_INTERVAL_SECONDS,
        )

    cycle_index += 1
    _sleep_until_next_boundary(POLL_INTERVAL_SECONDS)